In [1]:
# Install Earth Engine
!pip install -q geemap earthengine-api
!pip install -q arcgis geopandas

from arcgis.features import FeatureLayer
import geopandas as gpd
import ee
import geemap
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np


ee.Authenticate()  # You'll get a link to sign in with your Google account
ee.Initialize(project='thrive-care')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 21.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 738.5/738.5 kB 11.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 101.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 93.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 4.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.1 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_v

## Get Tensift basin from arcgis



In [2]:
basin_url = "https://services3.arcgis.com/hjUMsSJ87zgoicvl/arcgis/rest/services/RessourcesEau_Maroc/FeatureServer/3"
layer = FeatureLayer(basin_url)
geojson = layer.query(out_sr=4326).to_geojson
basin_gdf = gpd.read_file(geojson)

# --- Load Oueds layer ---
oueds_url = "https://services3.arcgis.com/hjUMsSJ87zgoicvl/arcgis/rest/services/RessourcesEau_Maroc/FeatureServer/0"
oueds_layer = FeatureLayer(oueds_url)
oueds_geojson = oueds_layer.query(out_sr=4326).to_geojson
oueds_gdf = gpd.read_file(oueds_geojson)

# --- Load Nappes layer ---
nappes_url = "https://services3.arcgis.com/hjUMsSJ87zgoicvl/arcgis/rest/services/RessourcesEau_Maroc/FeatureServer/1"
nappes_layer = FeatureLayer(nappes_url)
nappes_geojson = nappes_layer.query(out_sr=4326).to_geojson
nappes_gdf = gpd.read_file(nappes_geojson)

# --- Clip to Tensift basin ---
tensift = basin_gdf[basin_gdf['NomBassin'] == 'Tensift']
oueds_tensift = gpd.clip(oueds_gdf, tensift)
nappes_tensift = gpd.clip(nappes_gdf, tensift)
tensift_geom = geemap.geopandas_to_ee(tensift)


# Pond Site Suitability

In [3]:
import ee
import geemap
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Initialize Earth Engine
# ee.Initialize()

print("="*80)
print("POND SITE SUITABILITY ANALYSIS - TENSIFT BASIN (OPTIMIZED)")
print("="*80)
print("\nIdentifying optimal locations for water retention ponds")
print("All layers will be clipped to Tensift Basin boundary")
print("="*80)

# Helper function to create maps with legends
def create_map_with_legend(center_lat, center_lon, zoom=9):
    """Create a standardized map centered on Tensift"""
    m = geemap.Map(center=[center_lat, center_lon], zoom=zoom)
    m.add_basemap('HYBRID')
    return m

# ============================================================================
# TASK 2: LOAD DEM AND PREPROCESS
# ============================================================================

print("\n" + "="*80)
print("TASK 2: DEM PREPROCESSING & HYDROLOGICAL ANALYSIS")
print("="*80)

def load_and_process_dem(geometry):
    """Load SRTM DEM and derive hydrological parameters with HydroRIVERS streams"""
    try:
        print("\nLoading SRTM 30m DEM...")
        dem = ee.Image('USGS/SRTMGL1_003').select('elevation').clip(geometry)

        print("Calculating slope and flow accumulation...")
        dem_filled = dem.focal_min(radius=90, units='meters').blend(dem)
        slope = ee.Terrain.slope(dem_filled)

        # Simplified flow accumulation
        flow_acc_approx = dem_filled.focal_min(radius=500, units='meters')
        flow_acc = dem_filled.subtract(flow_acc_approx).abs().multiply(50)

        # Load HydroRIVERS stream network (better than DEM-based)
        print("Loading HydroRIVERS global river network...")
        try:
            hydrorivers = ee.FeatureCollection("WWF/HydroSHEDS/v1/FreeFlowingRivers")
            local_rivers = hydrorivers.filterBounds(geometry)

            # Convert to raster for analysis
            streams = ee.Image().byte().paint(local_rivers, 1, 100).selfMask()

            river_count = local_rivers.size().getInfo()
            print(f"  • Found {river_count} river segments in basin")

        except Exception as e:
            print(f"  ⚠️  HydroRIVERS failed, using DEM-based streams: {str(e)}")
            streams = flow_acc.gt(30).selfMask()

        print("✓ DEM preprocessing complete")

        return {
            'dem': dem,
            'dem_filled': dem_filled,
            'slope': slope,
            'flow_acc': flow_acc,
            'streams': streams,
            'rivers_fc': local_rivers if 'local_rivers' in locals() else None
        }
    except Exception as e:
        print(f"❌ Error: {str(e)}")
        raise

# Run Task 2
print("\nExecuting Task 2...")
dem_products = load_and_process_dem(tensift_geom)

# Visualization with proper clipping and legend
print("📊 Visualizing DEM products...")
Map1 = create_map_with_legend(31.5, -8, 9)
Map1.addLayer(tensift_geom, {'color': 'yellow'}, 'Tensift Basin', True, 0.5)
Map1.addLayer(dem_products['dem_filled'].clip(tensift_geom),
              {'min': 200, 'max': 4000, 'palette': ['#1a5f1a', '#90ee90', '#ffd700', '#8b4513', '#ffffff']},
              'Elevation (m)', True)
Map1.addLayer(dem_products['slope'].clip(tensift_geom),
              {'min': 0, 'max': 30, 'palette': ['#00ff00', '#ffff00', '#ff8800', '#ff0000', '#8b0000']},
              'Slope (degrees)', False)
Map1.addLayer(dem_products['streams'].clip(tensift_geom),
              {'palette': ['#0000ff']}, 'HydroRIVERS Stream Network', True)

# Add river features if available
if dem_products.get('rivers_fc'):
    Map1.addLayer(dem_products['rivers_fc'], {'color': 'darkblue'}, 'River Segments (vector)', False)

# Add custom legend
Map1.add_colorbar(
    vis_params={'min': 200, 'max': 4000, 'palette': ['#1a5f1a', '#90ee90', '#ffd700', '#8b4513', '#ffffff']},
    label='Elevation (m)',
    layer_name='Elevation (m)'
)
display(Map1)

# ============================================================================
# TASK 3: VALLEY BOTTOMS
# ============================================================================

print("\n" + "="*80)
print("TASK 3: VALLEY BOTTOM & DEPRESSION DETECTION")
print("="*80)

def extract_valley_bottoms(dem_products, geometry):
    """Identify valley bottoms - prime pond locations"""
    try:
        print("\nIdentifying valley bottoms...")

        slope = dem_products['slope']
        flow_acc = dem_products['flow_acc']
        dem = dem_products['dem_filled']

        # Low slope areas (< 5 degrees)
        low_slope = slope.lt(5)

        # High flow accumulation (simple threshold)
        high_flow = flow_acc.gt(40)

        # Topographic Position Index (valleys are negative)
        tpi = dem.subtract(dem.focal_mean(radius=300, units='meters'))
        valley_tpi = tpi.lt(-3)

        # Combine criteria
        valley_bottom_mask = low_slope.And(high_flow).Or(valley_tpi)
        valley_bottom_mask = valley_bottom_mask.focal_mode(radius=60, units='meters')

        print("✓ Valley bottom identification complete")

        return {
            'valley_bottom_mask': valley_bottom_mask,
            'low_slope': low_slope,
            'tpi': tpi
        }
    except Exception as e:
        print(f"❌ Error: {str(e)}")
        raise

print("\nExecuting Task 3...")
valley_data = extract_valley_bottoms(dem_products, tensift_geom)

print("📊 Visualizing valley bottoms...")
Map2 = create_map_with_legend(31.5, -8, 9)
Map2.addLayer(tensift_geom, {'color': 'yellow'}, 'Tensift Basin', True, 0.5)
Map2.addLayer(valley_data['valley_bottom_mask'].selfMask().clip(tensift_geom),
              {'palette': ['#00ff00']}, 'Valley Bottoms', True)
Map2.addLayer(valley_data['tpi'].clip(tensift_geom),
              {'min': -20, 'max': 20, 'palette': ['#0000ff', '#ffffff', '#ff0000']},
              'Topographic Position', False)
Map2.add_colorbar(
    vis_params={'min': -20, 'max': 20, 'palette': ['#0000ff', '#ffffff', '#ff0000']},
    label='TPI (negative = valleys)',
    layer_name='Topographic Position'
)
display(Map2)

# ============================================================================
# TASK 4: PRECIPITATION (SIMPLIFIED)
# ============================================================================

print("\n" + "="*80)
print("TASK 4: PRECIPITATION ANALYSIS (SIMPLIFIED)")
print("="*80)

def analyze_precipitation_runoff(geometry):
    """Load CHIRPS precipitation - simplified version"""
    try:
        print("\nLoading CHIRPS precipitation (2020-2024 for speed)...")

        chirps = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY') \
            .filterDate('2020-01-01', '2024-12-31') \
            .filterBounds(geometry)

        # Mean annual rainfall (simplified calculation)
        mean_annual_rain = chirps.select('precipitation').mean().multiply(365)

        # Simple runoff potential (30% of rainfall)
        runoff_potential = mean_annual_rain.multiply(0.3)

        print("✓ Precipitation analysis complete")

        return {
            'mean_annual_rain': mean_annual_rain,
            'runoff_potential': runoff_potential
        }
    except Exception as e:
        print(f"❌ Error: {str(e)}")
        raise

print("\nExecuting Task 4...")
precip_data = analyze_precipitation_runoff(tensift_geom)

print("📊 Visualizing precipitation...")
Map3 = create_map_with_legend(31.5, -8, 9)
Map3.addLayer(tensift_geom, {'color': 'yellow'}, 'Tensift Basin', True, 0.5)
Map3.addLayer(precip_data['mean_annual_rain'].clip(tensift_geom),
              {'min': 100, 'max': 600, 'palette': ['#8b0000', '#ff0000', '#ffa500', '#ffff00', '#00ff00', '#0000ff']},
              'Annual Rainfall (mm)', True)
Map3.add_colorbar(
    vis_params={'min': 100, 'max': 600, 'palette': ['#8b0000', '#ff0000', '#ffa500', '#ffff00', '#00ff00', '#0000ff']},
    label='Rainfall (mm/year)',
    layer_name='Annual Rainfall (mm)'
)
display(Map3)

# ============================================================================
# TASK 5: LANDCOVER & SOIL (SIMPLIFIED)
# ============================================================================

print("\n" + "="*80)
print("TASK 5: LANDCOVER & SOIL ANALYSIS")
print("="*80)

def analyze_landcover_soil(geometry):
    """Load landcover and soil data"""
    try:
        print("\nLoading ESA WorldCover...")
        landcover = ee.ImageCollection('ESA/WorldCover/v200').first().select('Map').clip(geometry)

        # Suitable: Grass(30), Crop(40), Bare(60), Shrub(20)
        suitable_lc = landcover.eq(30).Or(landcover.eq(40)).Or(landcover.eq(60)).Or(landcover.eq(20))
        unsuitable_lc = landcover.eq(50).Or(landcover.eq(80))  # Built, Water
        landcover_mask = suitable_lc.And(unsuitable_lc.Not())

        print("Loading SoilGrids...")
        sand = ee.Image("projects/soilgrids-isric/sand_mean").select('sand_0-5cm_mean').clip(geometry)
        clay = ee.Image("projects/soilgrids-isric/clay_mean").select('clay_0-5cm_mean').clip(geometry)

        sand_pct = sand.divide(10)
        clay_pct = clay.divide(10)

        # Infiltration classification: 1=Low (clayey), 2=Medium, 3=High (sandy)
        high_infil = sand_pct.gt(60)
        low_infil = clay_pct.gt(40)
        medium_infil = high_infil.Not().And(low_infil.Not())

        soil_infil_class = low_infil.multiply(1).add(medium_infil.multiply(2)).add(high_infil.multiply(3))

        # Runoff coefficient (higher clay = more runoff)
        runoff_coef = clay_pct.divide(100).clamp(0.2, 0.8)

        print("✓ Landcover and soil analysis complete")

        return {
            'landcover': landcover,
            'landcover_mask': landcover_mask,
            'sand_pct': sand_pct,
            'clay_pct': clay_pct,
            'soil_infil_class': soil_infil_class,
            'runoff_coef': runoff_coef
        }
    except Exception as e:
        print(f"❌ Error: {str(e)}")
        raise

print("\nExecuting Task 5...")
soil_data = analyze_landcover_soil(tensift_geom)

# Refine runoff with soil data
precip_data['runoff_potential'] = precip_data['mean_annual_rain'].multiply(soil_data['runoff_coef'])

print("📊 Visualizing soil properties...")
Map4 = create_map_with_legend(31.5, -8, 9)
Map4.addLayer(tensift_geom, {'color': 'yellow'}, 'Tensift Basin', True, 0.5)
Map4.addLayer(soil_data['soil_infil_class'].clip(tensift_geom),
              {'min': 1, 'max': 3, 'palette': ['#ff0000', '#ffff00', '#00ff00']},
              'Soil Infiltration', True)
Map4.addLayer(soil_data['clay_pct'].clip(tensift_geom),
              {'min': 0, 'max': 60, 'palette': ['#ffffcc', '#ffeda0', '#feb24c', '#f03b20', '#bd0026']},
              'Clay Content (%)', False)
Map4.addLayer(soil_data['landcover_mask'].selfMask().clip(tensift_geom),
              {'palette': ['#90ee90']}, 'Suitable Landcover', False)

Map4.add_colorbar(
    vis_params={'min': 1, 'max': 3, 'palette': ['#ff0000', '#ffff00', '#00ff00']},
    label='Soil Infiltration (1=Low, 3=High)',
    layer_name='Soil Infiltration'
)
display(Map4)

# ============================================================================
# TASK 6: INUNDATION FREQUENCY (SIMPLIFIED)
# ============================================================================

print("\n" + "="*80)
print("TASK 6: FLOOD/INUNDATION FREQUENCY (SIMPLIFIED)")
print("="*80)

def compute_inundation_frequency(geometry):
    """Simplified inundation analysis"""
    try:
        print("\nLoading Sentinel-1 (monthly composites for speed)...")

        # Use monthly composites to speed up
        s1 = ee.ImageCollection('COPERNICUS/S1_GRD') \
            .filterBounds(geometry) \
            .filterDate('2022-01-01', '2024-12-31') \
            .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
            .filter(ee.Filter.eq('instrumentMode', 'IW')) \
            .select('VV')

        # Monthly composites
        months = ee.List.sequence(0, 35)  # 36 months

        def monthly_composite(m):
            start = ee.Date('2022-01-01').advance(m, 'month')
            end = start.advance(1, 'month')
            monthly = s1.filterDate(start, end).median()
            return ee.Algorithms.If(
                s1.filterDate(start, end).size().gt(0),
                monthly.set('system:time_start', start.millis()),
                None
            )

        s1_monthly = ee.ImageCollection(months.map(monthly_composite).removeAll([None]))

        # Speckle filter and water detection
        def detect_water(image):
            filtered = image.focal_median(30, 'circle', 'meters')
            water = filtered.lt(-16).rename('water')
            return water

        s1_water = s1_monthly.map(detect_water)

        # Inundation frequency
        total_obs = s1_water.count()
        water_sum = s1_water.sum()
        inundation_frequency = water_sum.divide(total_obs).multiply(100)

        print("✓ Inundation analysis complete")

        return {
            'inundation_frequency': inundation_frequency
        }
    except Exception as e:
        print(f"❌ Error: {str(e)}")
        raise

print("\nExecuting Task 6...")
inundation_data = compute_inundation_frequency(tensift_geom)

print("📊 Visualizing inundation frequency...")
Map5 = create_map_with_legend(31.5, -8, 9)
Map5.addLayer(tensift_geom, {'color': 'yellow'}, 'Tensift Basin', True, 0.5)
Map5.addLayer(inundation_data['inundation_frequency'].clip(tensift_geom),
              {'min': 0, 'max': 40, 'palette': ['#ffffff', '#d0e5f5', '#7cb5d2', '#4292c6', '#084594']},
              'Inundation Frequency (%)', True)
Map5.add_colorbar(
    vis_params={'min': 0, 'max': 40, 'palette': ['#ffffff', '#d0e5f5', '#7cb5d2', '#4292c6', '#084594']},
    label='Inundation Frequency (%)',
    layer_name='Inundation Frequency (%)'
)
display(Map5)

# ============================================================================
# TASK 7: SUITABILITY ANALYSIS
# ============================================================================

print("\n" + "="*80)
print("TASK 7: POND SUITABILITY SCORING")
print("="*80)

def compute_pond_suitability(valley_data, precip_data, soil_data, inundation_data):
    """Combine layers into suitability score"""
    try:
        print("\nComputing suitability scores...")

        # Normalize inputs to 0-1
        valley_score = valley_data['valley_bottom_mask'].unmask(0).float()
        soil_score = soil_data['soil_infil_class'].subtract(1).divide(2).clamp(0, 1)
        runoff_score = precip_data['runoff_potential'].unitScale(0, 200).clamp(0, 1)

        # Ideal inundation: 10-30% (not too dry, not permanently flooded)
        inund = inundation_data['inundation_frequency']
        inund_ideal = ee.Image(20)  # 20% is ideal
        inund_score = inund.subtract(inund_ideal).abs().multiply(-1).add(20).divide(20).clamp(0, 1)

        lc_score = soil_data['landcover_mask'].unmask(0).float()
        slope_score = valley_data['low_slope'].unmask(0).float()

        # Weighted MCDA
        suitability = valley_score.multiply(0.25) \
            .add(soil_score.multiply(0.15)) \
            .add(runoff_score.multiply(0.20)) \
            .add(inund_score.multiply(0.15)) \
            .add(lc_score.multiply(0.15)) \
            .add(slope_score.multiply(0.10))

        suitability = suitability.clamp(0, 1).rename('suitability')

        print("✓ Suitability analysis complete")

        return {'suitability': suitability}
    except Exception as e:
        print(f"❌ Error: {str(e)}")
        raise

print("\nExecuting Task 7...")
suitability_data = compute_pond_suitability(valley_data, precip_data, soil_data, inundation_data)

print("📊 Visualizing suitability...")
Map6 = create_map_with_legend(31.5, -8, 9)
Map6.addLayer(tensift_geom, {'color': 'yellow'}, 'Tensift Basin', True, 0.5)
Map6.addLayer(suitability_data['suitability'].clip(tensift_geom),
              {'min': 0, 'max': 1, 'palette': ['#d73027', '#fc8d59', '#fee08b', '#d9ef8b', '#91cf60', '#1a9850']},
              'Pond Suitability Score', True)
Map6.add_colorbar(
    vis_params={'min': 0, 'max': 1, 'palette': ['#d73027', '#fc8d59', '#fee08b', '#d9ef8b', '#91cf60', '#1a9850']},
    label='Suitability (0=Poor, 1=Excellent)',
    layer_name='Pond Suitability Score'
)
display(Map6)

# ============================================================================
# TASK 8: EXTRACT CANDIDATES
# ============================================================================

print("\n" + "="*80)
print("TASK 8: EXTRACTING TOP CANDIDATE SITES")
print("="*80)

def extract_pond_candidates(suitability_data, dem_products, geometry, threshold=0.55):
    """Extract top candidate sites with distance to streams"""
    try:
        print(f"\nExtracting sites with suitability > {threshold}...")

        suitability = suitability_data['suitability']
        high_suitability = suitability.gte(threshold).selfMask()

        # Connected patches
        patch_size = high_suitability.connectedPixelCount(maxSize=500)
        large_patches = patch_size.gte(ee.Number(15)).selfMask()  # ~1.35 ha minimum

        # Calculate distance to nearest stream (important for pond water supply)
        streams = dem_products['streams']
        distance_to_stream = streams.fastDistanceTransform().sqrt() \
                                   .multiply(ee.Image.pixelArea().sqrt())  # Convert to meters

        print("  • Calculating distance to nearest stream...")

        # Sample candidate locations
        candidates_points = large_patches.addBands(suitability) \
            .addBands(dem_products['slope']) \
            .addBands(dem_products['dem_filled']) \
            .addBands(distance_to_stream.rename('dist_to_stream')) \
            .sample(
                region=geometry,
                scale=300,
                numPixels=100,
                seed=42,
                geometries=True
            )

        num_candidates = candidates_points.size().getInfo()
        print(f"✓ Found {num_candidates} candidate locations")

        # Calculate average distance to streams
        if num_candidates > 0:
            sample_data = candidates_points.getInfo()['features']
            distances = [f['properties'].get('dist_to_stream', 0) for f in sample_data]
            avg_dist = sum(distances) / len(distances) if distances else 0
            print(f"  • Average distance to stream: {avg_dist:.0f} meters")

        return {
            'high_suitability': high_suitability,
            'candidates_points': candidates_points,
            'num_candidates': num_candidates,
            'distance_to_stream': distance_to_stream
        }
    except Exception as e:
        print(f"❌ Error: {str(e)}")
        raise

print("\nExecuting Task 8...")
candidates = extract_pond_candidates(suitability_data, dem_products, tensift_geom, threshold=0.55)

print("📊 Visualizing candidate sites...")
Map7 = create_map_with_legend(31.5, -8, 9)
Map7.addLayer(tensift_geom, {'color': 'yellow'}, 'Tensift Basin', True, 0.5)
Map7.addLayer(dem_products['streams'].clip(tensift_geom),
              {'palette': ['#0000ff']}, 'HydroRIVERS Streams', True)
Map7.addLayer(suitability_data['suitability'].clip(tensift_geom),
              {'min': 0, 'max': 1, 'palette': ['#d73027', '#fc8d59', '#fee08b', '#d9ef8b', '#91cf60', '#1a9850']},
              'Suitability', False)
Map7.addLayer(candidates['high_suitability'].clip(tensift_geom),
              {'palette': ['#00ff00']}, '★ Top Candidates (>0.55)', True)
Map7.addLayer(candidates['candidates_points'], {'color': '#ff0000'}, '★ Sample Points', True)

# Add distance to stream layer
if candidates.get('distance_to_stream'):
    Map7.addLayer(candidates['distance_to_stream'].clip(tensift_geom),
                  {'min': 0, 'max': 5000, 'palette': ['blue', 'cyan', 'yellow', 'orange', 'red']},
                  'Distance to Stream (m)', False)

Map7.add_colorbar(
    vis_params={'min': 0, 'max': 1, 'palette': ['#d73027', '#fc8d59', '#fee08b', '#d9ef8b', '#91cf60', '#1a9850']},
    label='Suitability Score',
    layer_name='Suitability'
)
display(Map7)

# ============================================================================
# FINAL SUMMARY
# ============================================================================

print("\n" + "="*80)
print("ANALYSIS SUMMARY")
print("="*80)
print(f"\n📊 RESULTS:")
print(f"  • Candidate Pond Sites: {candidates['num_candidates']}")
print(f"  • Stream Network Source: HydroRIVERS (global database)")
print(f"  • Analysis Resolution: 30-300m depending on layer")
print(f"  • All layers clipped to Tensift Basin")

# Export candidate sites with attributes
print("\n💾 EXPORTING RESULTS...")
print("  Exporting candidate sites with attributes:")
print("    - Suitability score")
print("    - Elevation")
print("    - Slope")
print("    - Distance to nearest stream")

# Export to Google Drive
try:
    task = ee.batch.Export.table.toDrive(
        collection=candidates['candidates_points'],
        description='tensift_pond_candidates',
        fileFormat='GeoJSON'
    )
    task.start()
    print("\n✓ Export task started - check Google Drive")
except Exception as e:
    print(f"  ⚠️  Export setup: {str(e)}")

print("\n✅ ANALYSIS COMPLETE")
print("="*80)

POND SITE SUITABILITY ANALYSIS - TENSIFT BASIN (OPTIMIZED)

Identifying optimal locations for water retention ponds
All layers will be clipped to Tensift Basin boundary

TASK 2: DEM PREPROCESSING & HYDROLOGICAL ANALYSIS

Executing Task 2...

Loading SRTM 30m DEM...
Calculating slope and flow accumulation...
Loading HydroRIVERS global river network...
  • Found 1263 river segments in basin
✓ DEM preprocessing complete
📊 Visualizing DEM products...


Map(center=[31.5, -8], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(ch…


TASK 3: VALLEY BOTTOM & DEPRESSION DETECTION

Executing Task 3...

Identifying valley bottoms...
✓ Valley bottom identification complete
📊 Visualizing valley bottoms...


Map(center=[31.5, -8], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(ch…


TASK 4: PRECIPITATION ANALYSIS (SIMPLIFIED)

Executing Task 4...

Loading CHIRPS precipitation (2020-2024 for speed)...
✓ Precipitation analysis complete
📊 Visualizing precipitation...


Map(center=[31.5, -8], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(ch…

... (4 outputs left out)


TASK 7: POND SUITABILITY SCORING

Executing Task 7...

Computing suitability scores...
✓ Suitability analysis complete
📊 Visualizing suitability...


Map(center=[31.5, -8], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(ch…


TASK 8: EXTRACTING TOP CANDIDATE SITES

Executing Task 8...

Extracting sites with suitability > 0.55...
  • Calculating distance to nearest stream...
❌ Error: Computation timed out.


EEException: Computation timed out.